# E60 — 저장소 자체 체인(GPU)으로 최종 점수 재빌드

**런타임 → GPU 아무거나 (T4로 충분. 34B 모델을 돌리는 게 아니라 희소 ridge 회귀만 GPU로 푼다).**
**소요: 팔당 15~25분, 4팔 전부 1~1.5시간.**

지금까지의 숫자는 제 노트북에서 CUDA가 안 잡혀 선형 헤드를 CPU(scipy lsmr)로 대체한 체인에서 나왔다.
그래서 저장소가 발표한 **0.705568**과 직접 비교가 안 된다(같은 대체를 거친 재현값은 0.702727).

이 노트북은 **`tools/train_learned_router_gpu.py`를 진짜 cupy로 돌려** 대체 없이 같은 체인을 재현한다.
네 팔을 모두 같은 체인에서 재고, 첫 팔이 0.7055 근처면 체인이 맞는 것이다.

| 팔 | prior 구성 | CPU 대체 체인에서의 값 | 기대 |
|---|---|---:|---|
| `baseline` | 배포본 [A, B] | 0.702727 | **0.7055 근처면 체인 일치 확인** |
| `replace` | [A, C] ← 34B 컬럼 | 0.709290 | 최선 |
| `append` | [A, B, C] | 0.708722 | |
| `only` | [C] | 미측정 | 참고 |

C = 진짜 `skt/A.X-3.1`(34B) 컬럼, 39,507 엔트리, dev 커버리지 0.975, `corr(ax31) 0.709`.
이미 컴파일해서 번들에 넣었으므로(3.5 MB) 문항 풀 100 MB를 올릴 필요가 없다.

In [ ]:
#@title ① 번들 압축 해제
import os, zipfile, glob, shutil
BUNDLE = 'e60_colab_bundle.zip'
if not os.path.exists(BUNDLE):
    try:
        from google.colab import drive; drive.mount('/content/drive')
        c = glob.glob('/content/drive/MyDrive/**/' + BUNDLE, recursive=True)
        if c: shutil.copyfile(c[0], BUNDLE)
    except Exception as e: print('drive skip:', e)
if not os.path.exists(BUNDLE) or not zipfile.is_zipfile(BUNDLE):
    from google.colab import files; files.upload()
print('zip 정상:', zipfile.is_zipfile(BUNDLE), f'{os.path.getsize(BUNDLE)/1e6:.1f} MB')
with zipfile.ZipFile(BUNDLE) as z: z.extractall('.')
%cd /content/official-router
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title ② cupy 확인 — 저장소 체인은 GPU 선형 헤드를 쓴다
# Colab 에는 cupy 가 이미 있다. 없거나 CUDA 버전이 어긋나면 아래 한 줄을 주석 해제.
# !pip -q install cupy-cuda12x
import cupy as cp
print('cupy', cp.__version__)
d = cp.cuda.runtime.getDeviceProperties(0)
print('device:', d['name'].decode() if isinstance(d['name'], bytes) else d['name'])
print('sanity:', float(cp.arange(10, dtype=cp.float32).sum()))
import sklearn, scipy; print('sklearn', sklearn.__version__, '/ scipy', scipy.__version__)

In [ ]:
#@title ③ baseline — 배포본 그대로. 0.7055 근처면 체인이 맞다 (~20분)
!bash run_repo_chain.sh baseline 2>&1 | grep -v -i warn | tail -25

**③ 확인 포인트**
- `"training_backend": "gpu"` / `"solver": "cupyx-lsmr"` 가 찍혀야 진짜 저장소 체인이다.
- `final=` 이 **0.7055 근처**면 재현 성공. 크게 다르면 여기서 멈추고 출력을 그대로 알릴 것
  (그 경우 이후 팔의 비교 기준 자체가 흔들린다).

In [ ]:
#@title ④ replace — [A, C]. 34B 컬럼으로 교체 (~20분)
!bash run_repo_chain.sh replace 2>&1 | grep -v -i warn | tail -25

In [ ]:
#@title ⑤ append — [A, B, C] (~20분)
!bash run_repo_chain.sh append 2>&1 | grep -v -i warn | tail -25

In [ ]:
#@title ⑥ only — [C] 단독 (~20분, 참고용)
!bash run_repo_chain.sh only 2>&1 | grep -v -i warn | tail -25

In [ ]:
#@title ⑦ 요약표 + family별 내역
import json, glob, os
print(f"{'arm':<12}{'held-out dev':>14}{'fast':>9}{'balanced':>10}{'premium':>9}")
for arm in ('baseline', 'replace', 'append', 'only'):
    art = f'reports/repo_{arm}/learned-router.v1.json'
    if not os.path.exists(art): continue
    out = os.popen(f'PYTHONPATH=src python -X utf8 tools/holdout_eval.py --artifact {art} '
                   f'--input data/materialized/dev/inputs.json --outcomes data/dev/outcomes.json 2>/dev/null').read()
    fin = [l for l in out.splitlines() if l.startswith('final=')]
    tiers = {l.split(':')[0].strip(): l.split('score=')[1].split()[0] for l in out.splitlines() if 'score=' in l}
    if fin:
        print(f"{arm:<12}{float(fin[0][6:]):>14.6f}"
              f"{float(tiers.get('fast', 0)):>9.4f}{float(tiers.get('balanced', 0)):>10.4f}"
              f"{float(tiers.get('premium', 0)):>9.4f}")
print()
b, r = 'reports/repo_baseline/learned-router.v1.json', 'reports/repo_replace/learned-router.v1.json'
if os.path.exists(b) and os.path.exists(r):
    !PYTHONPATH=src python -X utf8 tools/holdout_by_family.py --artifact $b $r --labels baseline replace 2>&1 | tail -45

In [ ]:
#@title ⑧ 아티팩트 회수 → MyDrive (채택할 팔을 그대로 배포에 쓸 수 있게)
!cd /content/official-router && zip -qr /content/e60_out.zip reports/repo_*/ -x '*.bak'
!ls -la /content/e60_out.zip
import zipfile; print('zip 정상:', zipfile.is_zipfile('/content/e60_out.zip'))
from google.colab import drive; drive.mount('/content/drive')
!cp /content/e60_out.zip /content/drive/MyDrive/
import os
s, d = '/content/e60_out.zip', '/content/drive/MyDrive/e60_out.zip'
print('MyDrive 사본 크기 일치:', os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s))